In [78]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import os

In [79]:
import sys
from pathlib import Path

# Add project root (parent of the optimization folder) to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [80]:
THEATER_NAME = "AMC Boston Common 19"
NUM_SCREENS = 19
SCREENS = [f"Screen_{i}" for i in range(1, NUM_SCREENS+1)]

In [85]:
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MIN_GENRES = 5
OPENING_TIME = dt.time(11,0)
CLOSING_TIME=dt.time(23,0)
TICKET_PRICE = 11.31

# Price elasticity parameters
BASE_TICKET_PRICE = 11.31  # Reference price for demand calculations
PRICE_ELASTICITY = 1.2  # Demand decreases by 1.2% for every 1% price increase

# Genre diversity parameters
GENRE_DIVERSITY_WEIGHT = 0.05  # 5% demand boost per genre above minimum
MIN_GENRES_FOR_BONUS = 5  # Baseline - no penalty/bonus
MAX_GENRE_BONUS = 0.25  # Cap at 25% demand boost

# Intra-day repetition penalty (COST, not bonus - subtracts from objective)
# Penalizes showing same movie multiple times on the same day
REPETITION_PENALTY_PER_SHOWING = 200  # Dollar cost per showing beyond first on same day
# E.g., 1 showing = $0, 2 showings = $50, 3 showings = $100, etc.

TICKET_PRICES = [TICKET_PRICE]*len(DAYS)
today = dt.date.today()
open_dt = dt.datetime.combine(today, OPENING_TIME)
close_dt = dt.datetime.combine(today, CLOSING_TIME)
delta = close_dt - open_dt


OPERATING_MIN_PER_DAY = {
    day: delta.total_seconds()/60 for day in DAYS  # same for all days for now
}

SCREEN_CAPACITIES = {s: 210 
for s in SCREENS[:19]}
SCREEN_CAPACITIES["Screen_19"] = 600

BUFFER_MIN = 15

MAX_SHOWINGS_PER_MOVIE_PER_DAY = 4

# # Synthetic movie Data 
# movies_df = pd.DataFrame({
#     "movie_id": ["M1", "M2", "M3", "M4"],
#     "title": ["Blockbuster Action", "Indie Drama", "Family Animation", "Horror Thriller"],
#     "runtime_min": [130, 110, 95, 100],
#     "ticket_price": [15.0, 14.0, 13.0, 13.0]
# })

In [86]:
from tmdb_api_calling.utils import get_movies_out_now, get_movies_from_url

In [87]:
tmdb_key = os.getenv("TMDB_API_KEY")
headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {tmdb_key}"
}

In [88]:
# CURRENT_MOVIES = get_movies_out_now(headers)
movies_df = pd.read_csv("../data/cleaned/final_merged_dataset_with_genres.csv")

In [89]:
current = movies_df[movies_df['release_date'] > '2025-10-01']

In [90]:
current

,ticker,date,title,distributor,gross,percent_yd,percent_lw,theaters,per_theater,total_gross,...,history,horror,music,mystery,romance,science_fiction,thriller,tv_movie,war,western
267,Private,2025-11-06,Anniversary,Roadside Att…,23223,-0.50,-0.43,809.0,29.0,530320,...,0,0,0,0,0,0,0,0,0,0
475,CMCSA,2025-11-10,Black Phone 2,Universal,551845,-0.59,-0.27,2943.0,188.0,70558650,...,0,1,0,0,0,0,1,0,0,0
1306,LGF.A,2025-11-10,Good Fortune,Lionsgate,70339,-0.53,-0.64,746.0,94.0,16188087,...,0,0,0,0,0,0,0,0,0,0
1773,Private,2025-11-06,Kiss of the Spider Woman,Roadside Att…,48,-0.83,-0.97,11.0,4.0,1623221,...,0,0,0,0,0,0,0,0,0,0
1815,Private,2025-11-06,Last Days,Vertical Ent…,664,-0.46,-0.84,74.0,9.0,219915,...,0,0,0,0,0,0,0,0,0,0
2496,Private,2025-10-19,Re-Election,Picturehouse,2215,-0.40,1.01,2.0,1108.0,19222,...,0,0,0,0,0,0,0,0,0,0
2519,PARA,2025-11-10,Regretting You,Paramount Pi…,811523,-0.50,-0.23,3196.0,254.0,38936925,...,0,0,0,0,1,0,0,0,0,0
2585,PARA,2025-11-10,Roofman,Paramount Pi…,52015,-0.49,-0.61,545.0,95.0,22376641,...,0,0,0,0,0,0,0,0,0,0
2722,Private,2025-11-09,Shelby Oaks,Neon,25000,-0.19,-0.87,200.0,125.0,4400918,...,0,1,0,1,0,0,1,0,0,0
2836,SONY,2025-11-10,Soul on Fire,Sony Pictures,9079,-0.69,-0.55,275.0,33.0,7349877,...,0,0,0,0,0,0,0,0,0,0


In [91]:
current['weeks_in_release'] = current['weeks_in_release'].apply(lambda x: x if x > 0 else 1)

/var/folders/mq/p952ryqs1gv79cnvz9f05slr0000gn/T/ipykernel_89698/3178696047.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current['weeks_in_release'] = current['weeks_in_release'].apply(lambda x: x if x > 0 else 1)


In [92]:
current['weekly_gross_adjusted_per_theater'] = current['gross_per_theater_adjusted_2024'] / current['weeks_in_release']

/var/folders/mq/p952ryqs1gv79cnvz9f05slr0000gn/T/ipykernel_89698/66180017.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current['weekly_gross_adjusted_per_theater'] = current['gross_per_theater_adjusted_2024'] / current['weeks_in_release']


In [93]:
current['weekly_demand_per_theater'] = current.weekly_gross_adjusted_per_theater / TICKET_PRICE

/var/folders/mq/p952ryqs1gv79cnvz9f05slr0000gn/T/ipykernel_89698/473287358.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current['weekly_demand_per_theater'] = current.weekly_gross_adjusted_per_theater / TICKET_PRICE


In [94]:
current[[
    'title_key', 'date', 'gross', 'average_gross','popularity', 'weeks_in_release','is_weekend', 'total_gross_adjusted_2024', 'gross_per_theater_adjusted_2024','weekly_demand_per_theater'
]].head(10)

,title_key,date,gross,average_gross,popularity,weeks_in_release,is_weekend,total_gross_adjusted_2024,gross_per_theater_adjusted_2024,weekly_demand_per_theater
267,anniversary,2025-11-06,23223,2.322300e+04,0.0773,1,0,530320.0,655.525340,57.959800
475,black phone 2,2025-11-10,551845,2.762397e+06,166.8039,3,0,70558650.0,23975.076453,706.604081
1306,good fortune,2025-11-10,70339,6.715468e+05,0.0604,3,0,16188087.0,21699.848525,639.547555
1773,kiss of the spider woman,2025-11-06,48,2.047375e+04,3.6080,4,0,1623221.0,147565.545455,3261.837875
1815,last days,2025-11-06,664,2.519825e+04,2.1153,2,0,219915.0,2971.824324,131.380386
2496,re-election,2025-10-19,2215,2.462250e+03,1.4877,1,1,19222.0,9611.000000,849.778957
2519,regretting you,2025-11-10,811523,2.152311e+06,65.7274,2,0,38936925.0,12183.017835,538.594953
2585,roofman,2025-11-10,52015,7.018551e+05,15.9921,4,0,22376641.0,41058.056881,907.560939
2722,shelby oaks,2025-11-09,25000,2.614580e+05,14.1275,2,1,4400918.0,22004.590000,972.793546
2836,soul on fire,2025-11-10,9079,2.453708e+05,2.7327,4,0,7349877.0,26726.825455,590.778635


In [95]:
current.head()

,ticker,date,title,distributor,gross,percent_yd,percent_lw,theaters,per_theater,total_gross,...,horror,music,mystery,romance,science_fiction,thriller,tv_movie,war,western,weekly_demand_per_theater
267,Private,2025-11-06,Anniversary,Roadside Att…,23223,-0.50,-0.43,809.0,29.0,530320,...,0,0,0,0,0,0,0,0,0,57.959800
475,CMCSA,2025-11-10,Black Phone 2,Universal,551845,-0.59,-0.27,2943.0,188.0,70558650,...,1,0,0,0,0,1,0,0,0,706.604081
1306,LGF.A,2025-11-10,Good Fortune,Lionsgate,70339,-0.53,-0.64,746.0,94.0,16188087,...,0,0,0,0,0,0,0,0,0,639.547555
1773,Private,2025-11-06,Kiss of the Spider Woman,Roadside Att…,48,-0.83,-0.97,11.0,4.0,1623221,...,0,0,0,0,0,0,0,0,0,3261.837875
1815,Private,2025-11-06,Last Days,Vertical Ent…,664,-0.46,-0.84,74.0,9.0,219915,...,0,0,0,0,0,0,0,0,0,131.380386


# Optimization Formulation

1. Parameters

- $A_{i,d}$: base expected attendance for movie i per day d (before adjustments)
- $P$: ticket price
- $P_0$: base ticket price (reference)
- $\epsilon$: price elasticity of demand
- $D_i$: duration of movie i in minutes
- $G_i$: genres of movie i
- $B$: buffer time between shows
- $C_j$: capacity of screen j
- $H_{j,d}$: hours screen j operates on day d 
- $K$: maximum times a movie can be shown on a given day
- $L$: min number of genres per day
- $w_g$: genre diversity weight (bonus per genre)
- $L_0$: minimum genres for bonus calculation

**Demand Adjustments:**
- Price adjustment: $A'_{i,d} = A_{i,d} \times \left(\frac{P_0}{P}\right)^\epsilon$
  - Higher prices reduce demand (realistic elasticity)
  - $\epsilon = 1.2$ means 1% price increase → 1.2% demand decrease

2. Decision Variables:

- $s_{i,d} \in \{0, 1\}$: whether movie i is scheduled on day d
- $x_{i, d, j} \geq 0$: number of times we show movie i on screen j on day d (integer)
- $r_{i, d} \geq 0$: realized tickets served for movie i on day d (continuous)
- $g_{genre, d} \in \{0, 1\}$: whether genre is represented on day d

3. Objective function: Maximize total value (revenue + diversity bonus):

$$\max \left( \sum_{i, d} P \times r_{i,d} + \text{Diversity Bonus} \right)$$

Where:
$$\text{Diversity Bonus} = \bar{R} \times w_g \times \left(\frac{1}{|D|}\sum_{genre,d} g_{genre,d} - L_0\right)$$

- $\bar{R}$ = average daily revenue
- This incentivizes scheduling diverse genres for long-term customer satisfaction
- Each genre above baseline ($L_0$) contributes $w_g$ fraction of average revenue

4. Constraints
- Total screen time: $\sum_{j} x_{i,d,j} (D_i+B) \leq H_{d,j} \quad \forall d,j$
- Tickets served won't exceed capacity:  $r_{i,d}\leq \sum_{j} x_{i, d,j}\cdot C_j \quad \forall i,d$
- Can't sell more tickets than adjusted demand: $r_{i,d} \leq A'_{i,d} \quad \forall i,d$
- If not scheduled, no showings:  $\sum_{j} x_{i, d,j} \leq K \times s_{i,d} \quad \forall i,d$
- Each movie shown at most K times: $\sum_{j} x_{i, d,j} \leq K \quad \forall i,d$
- Genre linking (upper): $g_{genre,d} \leq \sum_{i \in I_{genre}} s_{i,d} \quad \forall genre, d$
- Genre linking (lower): $s_{i,d} \leq g_{genre,d} \quad \forall i \in I_{genre}, d$
- Minimum genres per day: $\sum_{genre} g_{genre, d} \geq L \quad \forall d$

### Enhanced Objective Function with Realistic Tradeoffs

The optimization model now includes two key enhancements that make sensitivity analysis more meaningful:

#### 1. Price-Demand Elasticity

**Problem:** Original model had ticket price always increasing revenue (unrealistic)

**Solution:** Demand now responds to price changes:
- Formula: `adjusted_demand = base_demand × (base_price / current_price)^elasticity`
- With elasticity = 1.2: A 10% price increase → 12% demand decrease
- Creates realistic tradeoff: Higher prices mean more revenue per ticket but fewer tickets sold

**Impact on sensitivity analysis:**
- Price sensitivity now shows an optimal price point (not just "higher is better")
- Captures diminishing returns from price increases

#### 2. Genre Diversity Bonus

**Problem:** No incentive for diverse programming; model would schedule only highest-revenue movies

**Solution:** Added diversity bonus to objective function:
- Represents customer satisfaction and repeat business from variety
- Formula: `bonus = avg_revenue × weight × (avg_genres_per_day - baseline)`
- Default: 5% revenue boost per genre above minimum

**Impact on sensitivity analysis:**  
- Genre diversity constraint now has measurable business value
- Creates tradeoff between short-term revenue maximization and long-term customer retention
- Makes MIN_GENRES parameter more interesting to analyze

#### Configurable Parameters

```python
BASE_TICKET_PRICE = 11.31      # Reference price
PRICE_ELASTICITY = 1.2          # Demand elasticity
GENRE_DIVERSITY_WEIGHT = 0.05   # 5% boost per extra genre
MIN_GENRES_FOR_BONUS = 5        # Baseline genre count
MAX_GENRE_BONUS = 0.25          # Cap at 25% total boost
```

These parameters can be tuned based on empirical data or used in sensitivity analysis to understand their impact.

### Genre Diversity Constraint Implementation

The genre diversity constraint ensures that each day features at least `MIN_GENRES` different genres, providing variety for moviegoers.

**Implementation approach:**
- **Genre-day binary variables:** `g[genre, d]` = 1 if genre is represented on day d, 0 otherwise
- **Linking constraints:** 
  - If any movie with a genre is scheduled, `g[genre, d] = 1`
  - If no movies with a genre are scheduled, `g[genre, d] = 0`
- **Diversity constraint:** $\sum_{genres} g[genre, d] \geq MIN\_GENRES$ for all days d

**Efficiency:** This approach adds only 19 genres × 7 days = 133 binary variables, avoiding the need for movie-genre-day combinations which would create thousands of variables.

**Handling multi-genre movies:** Since movies can have multiple genres, the linking constraints automatically handle this by setting `g[genre, d] = 1` for all genres that the scheduled movie contains.

### Preprocessing

In [96]:
# Fixed: Create proper demand structure with (movie_id, day) tuples
# and distribute weekly demand across days

runtimes = {}
demand = {}
movie_ids = []

# Daily demand distribution (weekends get more demand)
# These weights sum to 7 (representing 7 days)
daily_weights = {
    "Mon": 0.8,
    "Tue": 0.8,
    "Wed": 0.9,
    "Thu": 1.0,
    "Fri": 1.3,
    "Sat": 1.6,
    "Sun": 1.6
}

for (i, row) in current.iterrows():
    movie_id = i
    runtimes[movie_id] = row['runtime']
    # Get weekly demand per theater and convert to int
    weekly_demand = int(row['weekly_demand_per_theater'])
    
    # Distribute weekly demand across days based on weights
    for day in DAYS:
        # Each day gets its weighted share of weekly demand
        daily_demand = int(weekly_demand * daily_weights[day] / 7.0)
        demand[(movie_id, day)] = daily_demand
    
    movie_ids.append(movie_id)

print(f"Loaded {len(movie_ids)} movies")
print(f"Example demands for movie {movie_ids[0]}:")
for day in DAYS:
    print(f"  {day}: {demand[(movie_ids[0], day)]} tickets")

Loaded 14 movies
Example demands for movie 267:
  Mon: 6 tickets
  Tue: 6 tickets
  Wed: 7 tickets
  Thu: 8 tickets
  Fri: 10 tickets
  Sat: 13 tickets
  Sun: 13 tickets


In [97]:
def createModel(runtimes, demand, ticket_price, max_showings_per_day, operating_min_per_day, 
                screen_capacity, movie_ids, days, screens, buffer_min, 
                movie_genre_df=None, min_genres=None, 
                base_price=None, price_elasticity=None,
                genre_diversity_weight=None, min_genres_for_bonus=None, max_genre_bonus=None,
                repetition_penalty_per_showing=None,
                verbose=False):
    
    m = gp.Model("CinemaShowtimeScheduling")
    if not verbose:
        m.Params.OutputFlag = 0  # silence output by default
    
    # Adjust demand based on price elasticity
    adjusted_demand = {}
    if base_price is not None and price_elasticity is not None and base_price > 0:
        # Demand adjustment: demand_new = demand_base * (base_price / current_price)^elasticity
        price_ratio = base_price / ticket_price
        demand_multiplier = price_ratio ** price_elasticity
        for key, val in demand.items():
            adjusted_demand[key] = val * demand_multiplier
        if verbose:
            print(f"Price elasticity applied: {demand_multiplier:.3f}x demand adjustment")
    else:
        adjusted_demand = demand.copy()
    
    # Decision variables
    x = m.addVars(
        movie_ids, days, screens,
        vtype=GRB.INTEGER,
        lb=0,
        name="x"
    )
    
    # r[i,d] >= 0 continuous - realized tickets
    r = m.addVars(
        movie_ids, days,
        vtype=GRB.CONTINUOUS,
        lb=0.0,
        name="r"
    )
    
    # s[i,d] binary - whether movie is scheduled
    s = m.addVars(
        movie_ids, days,
        vtype=GRB.BINARY,
        name="s"
    )
    
    # Objective: maximize total revenue from ticket sales
    revenue_from_tickets = gp.quicksum(ticket_price * r[i, d] for i in movie_ids for d in days)
    
    # Add genre diversity bonus if enabled
    # This represents additional customer satisfaction and repeat business
    objective_expr = revenue_from_tickets
    g = None  # genre variables
    
    if (movie_genre_df is not None and genre_diversity_weight is not None and 
        min_genres_for_bonus is not None):
        
        # List of all genre columns
        genre_columns = ['action', 'adventure', 'animation', 'comedy', 'crime', 
                        'documentary', 'drama', 'family', 'fantasy', 'history', 
                        'horror', 'music', 'mystery', 'romance', 'science_fiction', 
                        'thriller', 'tv_movie', 'war', 'western']
        
        # Create genre-day binary variables
        g = m.addVars(genre_columns, days, vtype=GRB.BINARY, name="genre")
        
        # Calculate diversity bonus
        # Each genre above minimum contributes to customer satisfaction
        # Bonus = avg_weekly_revenue * diversity_weight * (genres - min_genres_for_bonus)
        avg_daily_revenue = revenue_from_tickets / len(days)
        total_genre_count = gp.quicksum(g[genre, d] for genre in genre_columns for d in days)
        avg_genres_per_day = total_genre_count / len(days)
        
        # Diversity bonus as a function of genres above baseline
        diversity_bonus = avg_daily_revenue * genre_diversity_weight * (avg_genres_per_day - min_genres_for_bonus)
        
        # Cap the bonus if specified
        if max_genre_bonus is not None:
            max_bonus_value = avg_daily_revenue * max_genre_bonus
            # Note: Can't directly cap in Gurobi, so we'll use the weight to implicitly cap
            # The weight should be calibrated such that max realistic genres don't exceed the cap
        
        objective_expr = objective_expr + diversity_bonus
        
        if verbose:
            print(f"Genre diversity bonus enabled: {genre_diversity_weight:.1%} per genre above {min_genres_for_bonus}")
    
    # Add intra-day repetition penalty (SUBTRACTS from objective)
    # Penalizes showing same movie multiple times on the same day
    if repetition_penalty_per_showing is not None and repetition_penalty_per_showing > 0:
        # Penalty = cost × (total_showings - number_of_scheduled_movie_days)
        # This penalizes each showing beyond the first per movie-day
        # E.g., if movie shown 3 times on Monday: penalty = cost × (3 - 1) = 2×cost
        
        total_showings = gp.quicksum(x[i, d, j] for i in movie_ids for d in days for j in screens)
        scheduled_movie_days = gp.quicksum(s[i, d] for i in movie_ids for d in days)
        
        repetition_penalty = repetition_penalty_per_showing * (total_showings - scheduled_movie_days)
        
        objective_expr = objective_expr - repetition_penalty
        
        if verbose:
            print(f"Intra-day repetition penalty: ${repetition_penalty_per_showing:.0f} per showing beyond first per day")
    
    m.setObjective(objective_expr, GRB.MAXIMIZE)
    
    # 1) Screen time constraint per screen & day
    for d in days:
        for j in screens:
            m.addConstr(
                gp.quicksum(x[i, d, j] * (runtimes[i] + buffer_min) for i in movie_ids)
                <= operating_min_per_day[d],
                name=f"Time_{d}_{j}"
            )
    
    # 2) Capacity: r[i,d] <= sum_j x[i,d,j] * capacity_j
    for i in movie_ids:
        for d in days:
            m.addConstr(
                r[i, d] <= gp.quicksum(x[i, d, j] * screen_capacity[j] for j in screens),
                name=f"Capacity_{i}_{d}"
            )
    
    # 3) Demand: r[i,d] <= adjusted_demand[i,d]
    for i in movie_ids:
        for d in days:
            demand_val = adjusted_demand.get((i, d), 0)
            m.addConstr(
                r[i, d] <= demand_val,
                name=f"Demand_{i}_{d}"
            )
    
    # 4) Frequency + schedule linking
    for i in movie_ids:
        for d in days:
            m.addConstr(
                gp.quicksum(x[i, d, j] for j in screens)
                <= max_showings_per_day * s[i, d],
                name=f"FreqLimit_{i}_{d}"
            )
    
    # 5) Genre diversity constraints (if enabled)
    if g is not None:
        genre_columns = ['action', 'adventure', 'animation', 'comedy', 'crime', 
                        'documentary', 'drama', 'family', 'fantasy', 'history', 
                        'horror', 'music', 'mystery', 'romance', 'science_fiction', 
                        'thriller', 'tv_movie', 'war', 'western']
        
        # Link genre variables to movie scheduling
        for genre in genre_columns:
            for d in days:
                # Get movies that have this genre
                movies_with_genre = [i for i in movie_ids 
                                    if i in movie_genre_df.index and 
                                    movie_genre_df.loc[i, genre] == 1]
                
                if movies_with_genre:
                    # If no movies with this genre are scheduled, g[genre,d] must be 0
                    m.addConstr(
                        g[genre, d] <= gp.quicksum(s[i, d] for i in movies_with_genre),
                        name=f"GenreUpper_{genre}_{d}"
                    )
                    
                    # If any movie with this genre is scheduled, g[genre,d] must be 1
                    for i in movies_with_genre:
                        m.addConstr(
                            s[i, d] <= g[genre, d],
                            name=f"GenreLower_{genre}_{d}_{i}"
                        )
        
        # Add diversity constraint: at least min_genres different genres per day
        if min_genres is not None and min_genres > 0:
            for d in days:
                m.addConstr(
                    gp.quicksum(g[genre, d] for genre in genre_columns) >= min_genres,
                    name=f"MinGenres_{d}"
                )
    
    # Return model AND variables so we can access them later
    return m, x, r, s, g

In [98]:
# Create model and get variable references
model, x, r, s, g = createModel(
    runtimes, demand, TICKET_PRICE, MAX_SHOWINGS_PER_MOVIE_PER_DAY, 
    OPERATING_MIN_PER_DAY, SCREEN_CAPACITIES, movie_ids, DAYS, SCREENS, 
    BUFFER_MIN, 
    movie_genre_df=current, 
    min_genres=MIN_GENRES,
    base_price=BASE_TICKET_PRICE,
    price_elasticity=PRICE_ELASTICITY,
    genre_diversity_weight=GENRE_DIVERSITY_WEIGHT,
    min_genres_for_bonus=MIN_GENRES_FOR_BONUS,
    max_genre_bonus=MAX_GENRE_BONUS,
    repetition_penalty_per_showing=REPETITION_PENALTY_PER_SHOWING,
    verbose=True
)

print(f"\nModel created with {len(movie_ids)} movies, {len(DAYS)} days, {len(SCREENS)} screens")
print(f"Genre diversity constraint: MIN_GENRES = {MIN_GENRES}")
print(f"Price elasticity: {PRICE_ELASTICITY}")
print(f"Genre diversity bonus: {GENRE_DIVERSITY_WEIGHT:.1%} per genre above {MIN_GENRES_FOR_BONUS}")
print(f"Repetition penalty: ${REPETITION_PENALTY_PER_SHOWING:.0f} per showing beyond first per day")
print(f"Total decision variables: {model.NumVars}")
print(f"Total constraints: {model.NumConstrs}")

Price elasticity applied: 1.000x demand adjustment
Genre diversity bonus enabled: 5.0% per genre above 5
Intra-day repetition penalty: $200 per showing beyond first per day

Model created with 14 movies, 7 days, 19 screens
Genre diversity constraint: MIN_GENRES = 5
Price elasticity: 1.2
Genre diversity bonus: 5.0% per genre above 5
Repetition penalty: $200 per showing beyond first per day
Total decision variables: 0
Total constraints: 0


In [99]:
# Test: Run optimization directly to check model
model.optimize()

print(f"\nOptimal objective value: ${model.objVal:,.2f}")
print(f"Number of non-zero variables: {sum(1 for v in model.getVars() if v.X > 1e-6)}")

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[rosetta2] - Darwin 24.2.0 24C2101)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 700 rows, 2191 columns and 6643 nonzeros (Max)
Model fingerprint: 0x7413a65c
Model has 2058 linear objective coefficients
Model has 13034 quadratic objective terms
Variable types: 98 continuous, 2093 integer (231 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [1e+01, 2e+02]
  QObjective range [2e-02, 2e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 4e+03]
Found heuristic solution: objective 19600.000000
Presolve removed 301 rows and 161 columns
Presolve time: 0.02s
Presolved: 5201 rows, 6832 columns, 20307 nonzeros
Variable types: 0 continuous, 6832 integer (1598 binary)

Root relaxation: objective 3.254312e+05, 314 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds     

In [100]:
def optimize_showtimes(model, x, r, s, movie_ids, days, screens, verbose=False):
    """
    Optimize the showtime scheduling model and extract results
    
    Args:
        model: Gurobi model
        x: Decision variables for showings per movie/day/screen
        r: Decision variables for realized tickets
        s: Binary variables for whether movie is scheduled
        movie_ids: List of movie IDs
        days: List of days
        screens: List of screen names
        verbose: Whether to print status
    """
    
    model.optimize()
    
    status_code = model.Status
    if status_code == GRB.OPTIMAL:
        status = "Optimal"
    elif status_code == GRB.INFEASIBLE:
        status = "Infeasible"
    elif status_code == GRB.UNBOUNDED:
        status = "Unbounded"
    else:
        status = f"Status_{status_code}"
    
    total_revenue = model.objVal if status_code == GRB.OPTIMAL else None
    
    if verbose:
        print("Status:", status)
        print(f"Total revenue: ${total_revenue:,.2f}" if total_revenue else "N/A")
    
    schedule_rows = []
    if status_code == GRB.OPTIMAL:
        for i in movie_ids:
            for d in days:
                for j in screens:
                    val = x[i, d, j].X
                    if val > 1e-6:
                        schedule_rows.append({
                            "movie_id": i,
                            "day": d,
                            "screen": j,
                            "showings": int(round(val))
                        })
    
    schedule_df = pd.DataFrame(schedule_rows)
    
    realized_rows = []
    if status_code == GRB.OPTIMAL:
        for i in movie_ids:
            for d in days:
                realized_rows.append({
                    "movie_id": i,
                    "day": d,
                    "realized_tickets": r[i, d].X,
                    "scheduled_flag": int(round(s[i, d].X))
                })
    realized_df = pd.DataFrame(realized_rows)
    
    return {
        "status": status,
        "total_revenue": total_revenue,
        "schedule_df": schedule_df,
        "realized_df": realized_df
    }

In [101]:
# Optimize and get results
results = optimize_showtimes(model, x, r, s, movie_ids, DAYS, SCREENS, verbose=True)

print(f"\n{'='*60}")
print("OPTIMIZATION RESULTS")
print(f"{'='*60}")
print(f"Status: {results['status']}")
print(f"Total Revenue: ${results['total_revenue']:,.2f}" if results['total_revenue'] else "N/A")
print(f"\nSchedule has {len(results['schedule_df'])} showtime slots")
print(f"Movies scheduled: {results['realized_df'][results['realized_df']['scheduled_flag'] == 1].shape[0]} movie-day combinations")

Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[rosetta2] - Darwin 24.2.0 24C2101)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 700 rows, 2191 columns and 6643 nonzeros (Max)
Model fingerprint: 0x7413a65c
Model has 2058 linear objective coefficients
Model has 13034 quadratic objective terms
Variable types: 98 continuous, 2093 integer (231 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [1e+01, 2e+02]
  QObjective range [2e-02, 2e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 4e+03]

MIP start from previous solve produced solution with objective 324575 (0.01s)
Loaded MIP start from previous solve with objective 324575

Presolve removed 301 rows and 161 columns
Presolve time: 0.03s
Presolved: 5201 rows, 6832 columns, 20307 nonzeros
Variable types: 0 continuous, 6832 integer (1598 binary)

Root relaxation: objective 3.254312e+05, 316 iterations, 0.0

In [102]:
# Analyze the schedule
print("\n" + "="*60)
print("SCHEDULE ANALYSIS")
print("="*60)

if len(results['schedule_df']) > 0:
    schedule_df = results['schedule_df']
    realized_df = results['realized_df']
    
    # Showings per movie
    print("\nShowings per movie:")
    showings_per_movie = schedule_df.groupby('movie_id')['showings'].sum().sort_values(ascending=False)
    for movie_id, count in showings_per_movie.items():
        movie_title = current.loc[movie_id, 'title'] if movie_id in current.index else f"Movie {movie_id}"
        print(f"  {movie_title}: {count} showings")
    
    # Tickets sold per movie
    print("\nTickets sold per movie:")
    tickets_per_movie = realized_df[realized_df['scheduled_flag'] == 1].groupby('movie_id')['realized_tickets'].sum().sort_values(ascending=False)
    for movie_id, tickets in tickets_per_movie.items():
        movie_title = current.loc[movie_id, 'title'] if movie_id in current.index else f"Movie {movie_id}"
        revenue = tickets * TICKET_PRICE
        print(f"  {movie_title}: {tickets:,.0f} tickets (${revenue:,.2f})")
    
    # Showings per day
    print("\nShowings per day:")
    showings_per_day = schedule_df.groupby('day')['showings'].sum()
    for day in DAYS:
        count = showings_per_day.get(day, 0)
        print(f"  {day}: {count} showings")
    
    # Screen utilization
    print("\nScreen utilization:")
    screens_used = schedule_df.groupby('screen')['showings'].sum().sort_values(ascending=False)
    print(f"  Screens used: {len(screens_used)} / {len(SCREENS)}")
    for screen, count in screens_used.head(10).items():
        print(f"  {screen}: {count} showings")
    
    # Genre diversity analysis
    print("\n" + "="*60)
    print("GENRE DIVERSITY ANALYSIS")
    print("="*60)
    
    genre_columns = ['action', 'adventure', 'animation', 'comedy', 'crime', 
                    'documentary', 'drama', 'family', 'fantasy', 'history', 
                    'horror', 'music', 'mystery', 'romance', 'science_fiction', 
                    'thriller', 'tv_movie', 'war', 'western']
    
    # Get scheduled movies per day
    scheduled_by_day = realized_df[realized_df['scheduled_flag'] == 1].groupby('day')['movie_id'].apply(list)
    
    print(f"\nGenres represented each day (MIN_GENRES = {MIN_GENRES}):")
    for day in DAYS:
        if day in scheduled_by_day.index:
            scheduled_movies = scheduled_by_day[day]
            # Get all genres represented
            genres_present = set()
            for movie_id in scheduled_movies:
                if movie_id in current.index:
                    for genre in genre_columns:
                        if current.loc[movie_id, genre] == 1:
                            genres_present.add(genre)
            print(f"  {day}: {len(genres_present)} genres - {sorted(genres_present)}")
        else:
            print(f"  {day}: 0 genres - []")
    
    # Display sample schedule
    print("\n" + "="*60)
    print("SAMPLE SCHEDULE (first 20 entries)")
    print("="*60)
    display(schedule_df.head(20))
    
else:
    print("\n⚠️ No schedule generated - check if demand is too low or constraints are too tight")


SCHEDULE ANALYSIS

Showings per movie:
  The Smashing Machine: 26 showings
  Kiss of the Spider Woman: 9 showings
  Black Phone 2: 7 showings
  Good Fortune: 7 showings
  Re-Election: 7 showings
  Regretting You: 7 showings
  Roofman: 7 showings
  Shelby Oaks: 7 showings
  Soul on Fire: 7 showings
  Stitch Head: 7 showings
  The Mastermind: 7 showings
  Truth & Treason: 7 showings
  Last Days: 4 showings

Tickets sold per movie:
  The Smashing Machine: 14,943 tickets ($169,005.33)
  Kiss of the Spider Woman: 3,718 tickets ($42,050.58)
  Truth & Treason: 1,123 tickets ($12,701.13)
  Shelby Oaks: 1,084 tickets ($12,260.04)
  Roofman: 1,033 tickets ($11,683.23)
  Re-Election: 969 tickets ($10,959.39)
  Black Phone 2: 803 tickets ($9,081.93)
  Good Fortune: 729 tickets ($8,244.99)
  Soul on Fire: 670 tickets ($7,577.70)
  Regretting You: 610 tickets ($6,899.10)
  Stitch Head: 244 tickets ($2,759.64)
  The Mastermind: 240 tickets ($2,714.40)
  Last Days: 100 tickets ($1,131.00)
  Anniversa

,movie_id,day,screen,showings
0,475,Mon,Screen_1,1
1,475,Tue,Screen_1,1
2,475,Wed,Screen_1,1
3,475,Thu,Screen_2,1
4,475,Fri,Screen_2,1
5,475,Sat,Screen_2,1
6,475,Sun,Screen_2,1
7,1306,Mon,Screen_1,1
8,1306,Tue,Screen_1,1
9,1306,Wed,Screen_1,1


In [103]:
# DIAGNOSTIC: Analyze intra-day repetition
print("="*60)
print("INTRA-DAY REPETITION DIAGNOSTIC")
print("="*60)

if len(results['schedule_df']) > 0:
    schedule_df = results['schedule_df']
    
    # Calculate showings per movie per day
    showings_per_movie_day = schedule_df.groupby(['movie_id', 'day'])['showings'].sum().reset_index()
    
    print("\nMovies with multiple showings on the same day:")
    repeated = showings_per_movie_day[showings_per_movie_day['showings'] > 1]
    
    if len(repeated) > 0:
        for _, row in repeated.iterrows():
            movie_title = current.loc[row['movie_id'], 'title']
            print(f"  {movie_title} on {row['day']}: {row['showings']} showings")
        
        # Calculate actual penalty
        total_showings = schedule_df['showings'].sum()
        scheduled_movie_days = showings_per_movie_day[showings_per_movie_day['showings'] > 0].shape[0]
        actual_penalty = REPETITION_PENALTY_PER_SHOWING * (total_showings - scheduled_movie_days)
        
        print(f"\nPenalty calculation:")
        print(f"  Total showings: {total_showings}")
        print(f"  Scheduled movie-days: {scheduled_movie_days}")
        print(f"  Excess showings: {total_showings - scheduled_movie_days}")
        print(f"  Penalty cost: ${actual_penalty:,.2f}")
        
    else:
        print("  None - each movie shown at most once per day")
    
    # Compare with baseline (if available)
    print(f"\n{'='*60}")
    print("COMPARISON")
    print(f"{'='*60}")
    print(f"Current objective value: ${results['total_revenue']:,.2f}")
    print(f"  (This includes genre bonus and penalty)")
    
    # Calculate pure revenue
    realized_df = results['realized_df']
    pure_revenue = realized_df['realized_tickets'].sum() * TICKET_PRICE
    print(f"Pure ticket revenue: ${pure_revenue:,.2f}")
    
    if len(repeated) > 0:
        print(f"Estimated penalty: -${actual_penalty:,.2f}")
        print(f"Net impact: ${results['total_revenue'] - pure_revenue:,.2f}")


INTRA-DAY REPETITION DIAGNOSTIC

Movies with multiple showings on the same day:
  Kiss of the Spider Woman on Sat: 2 showings
  Kiss of the Spider Woman on Sun: 2 showings
  The Smashing Machine on Fri: 4 showings
  The Smashing Machine on Mon: 3 showings
  The Smashing Machine on Sat: 4 showings
  The Smashing Machine on Sun: 4 showings
  The Smashing Machine on Thu: 4 showings
  The Smashing Machine on Tue: 3 showings
  The Smashing Machine on Wed: 4 showings

Penalty calculation:
  Total showings: 109
  Scheduled movie-days: 88
  Excess showings: 21
  Penalty cost: $4,200.00

COMPARISON
Current objective value: $324,575.31
  (This includes genre bonus and penalty)
Pure ticket revenue: $297,068.46
Estimated penalty: -$4,200.00
Net impact: $27,506.85


## Export Results for Sensitivity Analysis

In [104]:
# Create results directory if it doesn't exist
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# Export schedule
schedule_file = results_dir / "baseline_schedule.csv"
results['schedule_df'].to_csv(schedule_file, index=False)
print(f"✓ Schedule exported to: {schedule_file}")

# Export realized tickets/revenue
realized_file = results_dir / "baseline_realized.csv"
results['realized_df'].to_csv(realized_file, index=False)
print(f"✓ Realized tickets exported to: {realized_file}")

# Export summary statistics
summary_data = {
    'metric': [
        'total_revenue',
        'total_showings',
        'total_tickets_sold',
        'num_movies_scheduled',
        'num_screens_used',
        'avg_showings_per_movie',
        'avg_tickets_per_showing',
        'capacity_utilization'
    ],
    'value': [
        results['total_revenue'],
        len(results['schedule_df']),
        results['realized_df']['realized_tickets'].sum(),
        results['realized_df']['scheduled_flag'].sum(),
        results['schedule_df']['screen'].nunique(),
        results['schedule_df'].groupby('movie_id')['showings'].sum().mean(),
        results['realized_df']['realized_tickets'].sum() / results['schedule_df']['showings'].sum() if len(results['schedule_df']) > 0 else 0,
        results['realized_df']['realized_tickets'].sum() / (results['schedule_df']['showings'].sum() * 210) if len(results['schedule_df']) > 0 else 0  # assuming avg capacity 210
    ]
}
summary_df = pd.DataFrame(summary_data)
summary_file = results_dir / "baseline_summary.csv"
summary_df.to_csv(summary_file, index=False)
print(f"✓ Summary statistics exported to: {summary_file}")

# Export parameters used
params_data = {
    'parameter': [
        'ticket_price',
        'buffer_min',
        'max_showings_per_movie_per_day',
        'num_screens',
        'operating_hours_per_day',
        'num_movies',
        'num_days'
    ],
    'value': [
        TICKET_PRICE,
        BUFFER_MIN,
        MAX_SHOWINGS_PER_MOVIE_PER_DAY,
        NUM_SCREENS,
        OPERATING_MIN_PER_DAY['Mon'] / 60,  # convert to hours
        len(movie_ids),
        len(DAYS)
    ]
}
params_df = pd.DataFrame(params_data)
params_file = results_dir / "baseline_parameters.csv"
params_df.to_csv(params_file, index=False)
print(f"✓ Parameters exported to: {params_file}")

# Export movie data
movie_export = current[['title', 'runtime', 'weekly_demand_per_theater']].copy()
movie_export['movie_id'] = current.index
movie_file = results_dir / "movie_data.csv"
movie_export.to_csv(movie_file, index=False)
print(f"✓ Movie data exported to: {movie_file}")

# Export demand data (for sensitivity analysis)
demand_data = []
for (movie_id, day), demand_val in demand.items():
    demand_data.append({
        'movie_id': movie_id,
        'day': day,
        'demand': demand_val
    })
demand_df = pd.DataFrame(demand_data)
demand_file = results_dir / "demand_data.csv"
demand_df.to_csv(demand_file, index=False)
print(f"✓ Demand data exported to: {demand_file}")

print(f"\n{'='*60}")
print("All baseline results exported successfully!")
print(f"{'='*60}")

✓ Schedule exported to: results/baseline_schedule.csv
✓ Realized tickets exported to: results/baseline_realized.csv
✓ Summary statistics exported to: results/baseline_summary.csv
✓ Parameters exported to: results/baseline_parameters.csv
✓ Movie data exported to: results/movie_data.csv
✓ Demand data exported to: results/demand_data.csv

All baseline results exported successfully!
